In [1]:
import sys
sys.path.append('..')  # Add the parent directory to Python's search path
import pyomo.environ as pyo
import numpy as np
from pyomolayer import PyomoOptLayer
import torch
import time
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer

# SOCP
\begin{align}
\text{min}_\mathbf{x} & \frac{1}{2} \mathbf{x}^\top \mathbf{Q} \mathbf{x} + \mathbf{q}^\top\mathbf{x}, 
\\
\\
\text{s.t.} \quad & \| \mathbf{x} ||^2_2 \leq 1  \\
& \mathbf{F} \mathbf{x} = \mathbf{g}
\end{align}
with the variable $\mathbf{x}$ and parameters $\mathbf{F} \in \mathbf{R}^{p\times n}$ and $\mathbf{A_i} \in \mathbf{R}^{n_i\times n}$.

In [2]:
def create_model(nominal_Psqrt, nominal_q, nominal_G, nominal_h):
    # Create a concrete model
    m = pyo.ConcreteModel()

    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    m.ipopt_zL_out = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    m.ipopt_zU_out = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    # Define variables
    m.x = pyo.Var(range(n), within=pyo.Reals)
    
    # Define parameters
    m.Psqrt = pyo.Var(range(n), range(n), within=pyo.Reals)
    m.q = pyo.Var(range(n), within=pyo.Reals)            # Linear term vector
    m.G = pyo.Var(range(p), range(n), within=pyo.Reals)  # Inequality constraint matrix
    m.h = pyo.Var(range(p), within=pyo.Reals) 
    
    for i in range(n):
        for j in range(n):
            m.Psqrt[i, j].fix(nominal_Psqrt[i, j])

    for i in range(n):
        m.q[i].fix(nominal_q[i])

    for i in range(p):
        for j in range(n):
            m.G[i, j].fix(nominal_G[i, j])

    for i in range(p):
        m.h[i].fix(nominal_h[i])

    
    # Define equality constraints
    m.equ_constraints = pyo.ConstraintList()
    for i in range(p):
        m.equ_constraints.add(sum(m.G[i, j] * m.x[j] for j in range(n)) == m.h[i])

    # Define inequality constraints
    #Norm    
    m.inequ_constraints = pyo.ConstraintList()

    m.inequ_constraints.add((sum(m.x[i]**2 for i in range(n))) <= 1)
    
    # Define objective function: 
    def objective_rule(m):
        tol_term = 0
        for i in range(n):
            q_term = 0
            for j in range(n):
                q_term += m.Psqrt[i, j] * m.x[j]
            tol_term += q_term**2
            
        return 0.5 * tol_term + sum(m.q[i] * m.x[i] for i in range(n))
        
    m.obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)
     
    return m

n, p = 8, 2

nominal_Psqrt = np.random.rand(n, n)
nominal_q = np.random.rand(n)

nominal_G = np.random.rand(p, n)
nominal_h = np.random.rand(p)
model = create_model(nominal_Psqrt, nominal_q, nominal_G, nominal_h)
opt = pyo.SolverFactory('ipopt', tee=True)
results = opt.solve(model) 
print(results)
#model.pprint()
print(model.obj())


Problem: 
- Lower bound: -inf
  Upper bound: inf
  Number of objectives: 1
  Number of constraints: 3
  Number of variables: 8
  Sense: unknown
Solver: 
- Status: ok
  Message: Ipopt 3.13.2\x3a Optimal Solution Found
  Termination condition: optimal
  Id: 0
  Error rc: 0
  Time: 0.1277017593383789
Solution: 
- number of solutions: 0
  number of solutions displayed: 0

-0.006758631721645786


In [3]:
variables_name = ["x"]

variables_size = {'x':[n]}

parameters_name = ["Psqrt", "q", "G", "h"]

parameters_size = {'Psqrt':[n, n], "q" : [n], 'G':[p, n], "h": [p]}

Pyomolayer = PyomoOptLayer(create_model, variables_name, variables_size, parameters_name, parameters_size, solver = 'ipopt')

In [4]:
torch.manual_seed(0)
batch_size = 10
Psqrtval = torch.randn(batch_size, n, n, requires_grad=True, dtype=torch.float64)
qval = torch.randn(batch_size, n, requires_grad=True, dtype=torch.float64)
Gval = torch.randn(batch_size, p, n, requires_grad=True, dtype=torch.float64)
hval= torch.randn(batch_size, p, requires_grad=True, dtype=torch.float64)


In [5]:
start = time.time()

input = tuple([Psqrtval, qval, Gval, hval])
primal, _, _, _ = Pyomolayer(*input)
primal.sum().backward()

end = time.time()

pyomo_time = end - start

print("pyomo_time", pyomo_time)

pyomo_time 0.7965724468231201


In [6]:
x = cp.Variable(n)

P_sqrt = cp.Parameter((n, n), name='P_sqrt')
q = cp.Parameter(n, name='f')
G = cp.Parameter((p, n), name='F')
h = cp.Parameter(p, name='g')

objective = 0.5 * cp.sum_squares(P_sqrt @ x) + q.T @ x
constraints = [G@x == h, cp.sum_squares(x) <= 1]
prob = cp.Problem(cp.Minimize(objective), constraints)

Layer = CvxpyLayer(prob, [P_sqrt, q, G, h], [x])

torch.manual_seed(0)
Psqrtval_cvx = torch.randn(batch_size, n, n, requires_grad=True, dtype=torch.float64)
qval_cvx = torch.randn(batch_size, n, requires_grad=True, dtype=torch.float64)
Gval_cvx = torch.randn(batch_size, p, n, requires_grad=True, dtype=torch.float64)
hval_cvx = torch.randn(batch_size, p, requires_grad=True, dtype=torch.float64)

In [7]:
start = time.time()

primal_cvx, = Layer(Psqrtval_cvx, qval_cvx, Gval_cvx, hval_cvx)

primal_cvx.sum().backward()

end = time.time()

cvxpy_time = end - start

print("cvxpy_time", cvxpy_time)

cvxpy_time 0.14870142936706543


In [8]:
print("Primal", np.max(torch.abs(primal_cvx - primal).detach().numpy()))
print("Grad_Psqrt", np.max(torch.abs(Psqrtval_cvx.grad - Psqrtval.grad).detach().numpy()))
print("Grad_q", np.max(torch.abs(qval_cvx.grad - qval.grad).detach().numpy()))
print("Grad_G", np.max(torch.abs(Gval_cvx.grad - Gval.grad).detach().numpy()))
print("Grad_h", np.max(torch.abs(hval_cvx.grad - hval.grad).detach().numpy()))

Primal 2.6304209558181135e-05
Grad_Psqrt 4.7127840885369254e-05
Grad_q 1.2940252830218046e-05
Grad_G 2.2636921552937617e-05
Grad_h 1.1837273210302257e-05
